# 11 — Role Intelligence + the O*NET Crosswalk

**Day 3, Step 11 — the keystone.**

Finding F4: **zero** of the 9 attrition job roles and zero of the 31 engagement
titles match an O*NET occupation title exactly. Without a crosswalk the three
O*NET files are inert and the entire skills half of the project has no input.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


In [2]:
from hrai.skills.crosswalk import distinct_roles, resolved_crosswalk

roles = distinct_roles()
occ = load_processed("occupation_master")
print(f"{len(roles)} distinct roles across both populations")
print(f"{len(occ):,} O*NET occupations")
print("exact title matches:",
      len(set(roles['role']) & set(occ['occupation_title'])))

40 distinct roles across both populations
1,016 O*NET occupations
exact title matches: 0


## Semantic matching, then human review

A sentence-transformer embeds both vocabularies and matches on cosine
similarity, so "Sr. DBA" can reach "Database Administrators" despite sharing no
words. But the automatic result is not trustworthy on its own — it mapped **CIO
to *Editors*** at 0.21 confidence, because a three-letter acronym carries almost
no semantic signal.

40 roles is small enough to check by hand, so all 40 were reviewed into
`conf/role_crosswalk_reviewed.yaml`, each with a written reason.

In [3]:
cw = resolved_crosswalk()
print(f"roles: {len(cw)}  human-reviewed: {int(cw['reviewed'].sum())}  "
      f"still flagged: {int(cw['needs_review'].sum())}")
cw[["role", "soc_code", "occupation_title", "confidence", "reviewed"]].head(15)

2026-08-28 01:55:04 | INFO  | crosswalk resolved


roles: 40  human-reviewed: 40  still flagged: 0


,role,soc_code,occupation_title,confidence,reviewed
0,Production Technician I,51-9061.00,"Inspectors, Testers, Sorters, Samplers, and We...",0.5570,True
1,Production Technician II,17-3026.00,Industrial Engineering Technologists and Techn...,0.5306,True
2,Sales Executive,41-4011.00,"Sales Representatives, Wholesale and Manufactu...",0.5727,True
3,Area Sales Manager,11-2022.00,Sales Managers,0.6138,True
4,Research Scientist,19-1042.00,"Medical Scientists, Except Epidemiologists",0.5909,True
5,Laboratory Technician,29-2012.00,Medical and Clinical Laboratory Technicians,0.6891,True
6,Production Manager,11-3051.00,Industrial Production Managers,0.6318,True
7,Manufacturing Director,11-3051.00,Industrial Production Managers,0.4622,True
8,Healthcare Representative,41-4011.00,"Sales Representatives, Wholesale and Manufactu...",0.6860,True
9,Manager,11-1021.00,General and Operations Managers,0.6515,True


### What the review corrected

| Role | Automatic match | Reviewed to |
|---|---|---|
| CIO | *Editors* (0.21) | Computer and Information Systems Managers |
| Senior BI Developer | *Editors* (0.36) | Business Intelligence Analysts |
| BI Director | *Media Technical Directors* | Computer and Information Systems Managers |
| Laboratory Technician | *Dental Laboratory Technicians* | Medical and Clinical Laboratory Technicians |
| Shared Services Manager | *Personal Service Managers* | General and Operations Managers |
| Sales Executive | *Sales Managers* | Sales Representatives (technical products) |

In [4]:
from hrai.skills.ontology import role_requirement_summary
summary = role_requirement_summary()
print("every role has skill data:",
      summary["technical_skills"].notna().all() and summary["foundational_skills"].notna().all())
summary.head(12)

2026-08-28 01:55:04 | INFO  | crosswalk resolved


2026-08-28 01:55:04 | INFO  | role requirements built


every role has skill data: True


,role,soc_code,occupation_title,technical_skills,hot_technologies,in_demand_tools,foundational_skills,mean_required_level
0,Accountant I,13-2011.00,Accountants and Auditors,25,25,6,9,3.51
1,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Exc...",25,25,5,10,2.81
2,Area Sales Manager,11-2022.00,Sales Managers,25,25,5,10,3.59
3,BI Developer,15-2051.01,Business Intelligence Analysts,25,25,14,10,3.55
4,BI Director,11-3021.00,Computer and Information Systems Managers,25,24,3,10,3.61
5,CIO,11-3021.00,Computer and Information Systems Managers,25,24,3,10,3.61
6,Data Analyst,15-2051.01,Business Intelligence Analysts,25,25,14,10,3.55
7,Data Architect,15-1243.00,Database Architects,25,25,13,10,3.46
8,Database Administrator,15-1242.00,Database Administrators,25,25,25,10,3.64
9,Director of Operations,11-1021.00,General and Operations Managers,25,25,3,10,3.44
